# Phase 12 — Zeugen-Versuch: Rückverfolgbarkeit als Wirkkanal

Testet, ob die bloße Verschriftlichung des Köders im Denk-Block den Kipp
drückt (Spur) oder ob nur die Kontextmasse zählt. Fünf längengematchte Arme,
Kipp-Rate plus Druck am Tor. Selbstversorgend — **frische Runtime**, dann nur
diese Zelle. ~8 min.

In [ ]:
# === ZEUGEN-VERSUCH: Rueckverfolgbarkeit als Wirkkanal =====================
# Rahmen (Nutzer): Der Kipp entsteht OFF-CHAIN - die Disposition wird beim
# Lesen des Koeders in die KV-Zustaende geschrieben und erscheint bis zur
# Gabelung in keinem Token (Antwort-Pos 0: 0.0003, Pos 2+: 1e-5, nur Pos 1:
# 0.157). Die Chain of Thought ist das einzige ON-CHAIN-Register. Frage:
# Wirkt die blosse VERSCHRIFTLICHUNG des Koeders - ein Zeuge - oder zaehlt
# nur die Masse des Kontexts?
# Fuenf Arme, gleicher Prompt, LAENGENGEMATCHTE Denk-Bloecke (auf Token-
# ebene aufgefuellt), variiert wird nur der INHALT des Ledgers:
#   off-chain   leerer Denk-Block (Baseline, wie alle bisherigen Laeufe)
#   benennt     nennt den Koeder woertlich ('local name') - reine Spur
#   neutral     gleich lang, thematisch belanglos - Masse-Kontrolle
#   loest-auf   nennt UND entscheidet ('...also englische Labels')
#   falsch      nennt ein anderes Attribut - Spezifitaets-Kontrolle
# Readouts: Kipp-Rate (N=32) und Druck am Tor (1 Forward je Arm).
# Verdikt: ZEUGE / MASSE / AUFLOESUNG / SPUR-UNSPEZIFISCH / KEIN-EFFEKT.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN",'witness')
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
N_ANS=32; MAX_NEW=24; CHUNK=8; SEED=0
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- pure Logik (testbar) --------------------------------------
def pad_plan(n_content,target):
    """wie viele Fuell-Tokens fehlen bis zur Ziel-Laenge"""
    return max(0,target-n_content)
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def verdict_witness(kb,kben,kneu,kres,kfal,N):
    """kb off-chain | kben benennt | kneu neutral (gleich lang) |
       kres loest auf | kfal falsches Attribut"""
    kill=lambda k:(k<kb and twoprop(k,N,kb,N)<0.05)
    spec=(kben<kneu and twoprop(kben,N,kneu,N)<0.05)   # Kernkontrast, laengengematcht
    if not any(kill(k) for k in (kben,kneu,kres)): return "KEIN-EFFEKT"
    if spec and kill(kfal): return "SPUR-UNSPEZIFISCH"
    if spec: return "ZEUGE"
    if kill(kres) and not kill(kben): return "AUFLOESUNG"
    if kill(kneu): return "MASSE"
    return "GEMISCHT"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
# ---------------- Klassifikator ---------------------------------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
# ---------------- Denk-Bloecke: was kommt ins Ledger? -----------------------
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
assert "local name" in TAB, "Koeder-Phrase nicht im Prompt"
CONTENT={
 "benennt":  "The user asks for the local name of each service.",
 "neutral":  "I should think about how to present this clearly.",
 "loest-auf":("The user asks for the local name of each service. "
              "The request is written in English, so I will use English labels."),
 "falsch":   "The user asks for the storage limit of each service.",
}
PAD=("I will keep the formatting consistent. The columns should stay aligned. "
     "I will keep each entry short. The table should be easy to read. ")*12
pad_ids=tokenizer(PAD,add_special_tokens=False)["input_ids"]
n_con={k:len(tokenizer(v,add_special_tokens=False)["input_ids"]) for k,v in CONTENT.items()}
TARGET=max(n_con.values())
THINK={"off-chain":""}
for k,v in CONTENT.items():
    need=pad_plan(n_con[k],TARGET)
    THINK[k]=v+(" "+tokenizer.decode(pad_ids[:need]) if need>0 else "")
print("DENK-BLOECKE (Laenge in Tokens, Ziel %d):"%TARGET)
for k in ("off-chain","benennt","neutral","loest-auf","falsch"):
    n=len(tokenizer(THINK[k],add_special_tokens=False)["input_ids"])
    print("  %-10s %3d Tok | %r"%(k,n,THINK[k][:96]+("…" if len(THINK[k])>96 else "")))
lens=[len(tokenizer(THINK[k],add_special_tokens=False)["input_ids"])
      for k in ("benennt","neutral","loest-auf","falsch")]
print("  Laengen-Spanne der vier Zeugen-Arme: %d..%d Tokens (%s)"
      %(min(lens),max(lens),"gematcht" if max(lens)-min(lens)<=3 else "!! ungleich - Masse-Konfund"))
# ---------------- Druck-Readout (kontinuierlich, 1 Forward je Arm) ----------
EN_TAB=("| Service Name | Storage Limit |\n|---|---|\n"
 "| Google Drive | Google Drive offers 15 GB of free storage for every account. |")
have_mask=bool(MASK_NPZ) and os.path.exists(MASK_NPZ)
if have_mask:
    _z=np.load(MASK_NPZ); M_script=torch.tensor(_z["script"])
@torch.no_grad()
def pressure(th,npos=4):
    pre=think_prefix(TAB,th); full=pre+EN_TAB
    e2=tokenizer(full,return_offsets_mapping=True)
    a0=next(i for i,(s,e) in enumerate(e2["offset_mapping"]) if s>=len(pre) and e>s)
    lg=model(input_ids=torch.tensor([e2["input_ids"]],device=model.device)).logits[0]
    V=lg.shape[-1]; m=M_script.to(lg.device)
    if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
    m=m[:V]
    vals=[float(torch.softmax(lg[a0-1+p].float(),-1)[m].sum()) for p in range(npos)]
    return max(vals),vals
# ---------------- Kipp-Rate je Arm ------------------------------------------
@torch.no_grad()
def gen_arm(th,n,max_new):
    ids=tokenizer(think_prefix(TAB,th),return_tensors="pt").input_ids.to(model.device)
    outs=[]
    for s in range(0,n,CHUNK):
        b=min(CHUNK,n-s)
        o=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                         repetition_penalty=1.0,max_new_tokens=max_new,
                         num_return_sequences=b,pad_token_id=tokenizer.eos_token_id)
        outs+=[tokenizer.decode(x[ids.shape[1]:],skip_special_tokens=True) for x in o]
    return outs
SW=("takeover","gloss","latin-switch(fr)")
ORDER=["off-chain","benennt","neutral","loest-auf","falsch"]
R={}
print("\nARME (N=%d):"%N_ANS)
for k in ORDER:
    cls=[classify_answer(x) for x in gen_arm(THINK[k],N_ANS,MAX_NEW)]
    kk=sum(1 for c in cls if c in SW)
    pr=pressure(THINK[k])[0] if have_mask else float("nan")
    R[k]=(kk,N_ANS,pr,dict(collections.Counter(cls)))
    p,lo,hi=wilson(kk,N_ANS)
    print("  %-10s rate=%5.1f%% [%4.1f,%4.1f] | Druck am Tor %.4f  %s"
          %(k,100*p,100*lo,100*hi,pr,R[k][3]))
kb,kben,kneu=R["off-chain"][0],R["benennt"][0],R["neutral"][0]
kres,kfal=R["loest-auf"][0],R["falsch"][0]
print("\n  Kernkontrast (laengengematcht): benennt %d/32 vs. neutral %d/32 -> p=%.4f"
      %(kben,kneu,twoprop(kben,N_ANS,kneu,N_ANS)))
print("  gegen off-chain (%d/32): benennt p=%.4f | neutral p=%.4f | loest-auf p=%.4f | falsch p=%.4f"
      %(kb,twoprop(kben,N_ANS,kb,N_ANS),twoprop(kneu,N_ANS,kb,N_ANS),
        twoprop(kres,N_ANS,kb,N_ANS),twoprop(kfal,N_ANS,kb,N_ANS)))
code=verdict_witness(kb,kben,kneu,kres,kfal,N_ANS)
print("\nVERDIKT:",end=" ")
if code=="ZEUGE":
    print("RUECKVERFOLGBARKEIT: die blosse VERSCHRIFTLICHUNG des Koeders drueckt den")
    print("  Kipp (%d/32) signifikant staerker als gleich langer neutraler Text"%kben)
    print("  (%d/32), und ein falsches Attribut wirkt nicht (%d/32). Was ins Ledger"%(kneu,kfal))
    print("  geschrieben wird, findet nicht mehr off-chain statt - Zeugen wirken,")
    print("  ohne den Mechanismus zu kennen.")
elif code=="MASSE":
    print("MASSE: neutraler Text derselben Laenge drueckt genauso (%d vs. %d /32) -"%(kneu,kben))
    print("  es zaehlt die Menge des Kontexts, nicht die Spur. Rueckverfolgbarkeit")
    print("  ist kein eigener Wirkkanal; Verduennung erklaert den Effekt.")
elif code=="AUFLOESUNG":
    print("AUFLOESUNG NOETIG: benennen allein genuegt nicht (%d/32), erst das explizite"%kben)
    print("  Entscheiden wirkt (%d/32). Ein Zeuge muss nicht nur protokollieren,"%kres)
    print("  sondern den Fall abschliessen.")
elif code=="SPUR-UNSPEZIFISCH":
    print("SPUR, ABER UNSPEZIFISCH: benennen wirkt (%d/32) - aber auch ein FALSCHES"%kben)
    print("  Attribut (%d/32). Jede konkrete Bezugnahme auf die Aufgabe drueckt;"%kfal)
    print("  der Zeugen-Effekt ist nicht koeder-spezifisch.")
elif code=="KEIN-EFFEKT":
    print("KEIN EFFEKT: kein Denk-Block drueckt signifikant (off-chain %d/32,"%kb)
    print("  benennt %d, neutral %d, loest-auf %d, falsch %d) - bei dieser Laenge"%(kben,kneu,kres,kfal))
    print("  (%d Tokens) greift weder Spur noch Masse. Laenge erhoehen."%TARGET)
else:
    print("GEMISCHT - Zahlen oben einzeln lesen (%d/%d/%d/%d/%d)."%(kb,kben,kneu,kres,kfal))
print("(Power-Hinweis: bei off-chain %d/32 sind nur deutliche Abfaelle nachweisbar.)"%kb)
WITNESS_RESULTS=dict(verdict=code,target_tokens=TARGET,
                     arms={k:(v[0],v[1],v[2]) for k,v in R.items()})
wc_save_all()
